In [1]:
!pip install scikeras

In [11]:
#Importing the packages
import pandas as pd
import numpy as np
import cv2
import keras.models as M
import keras.layers as L
import os
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import clear_output
import keras

In [18]:
import os
import numpy as np
import rasterio
from rasterio import warp
from rasterio.enums import Resampling
import matplotlib.pyplot as plt

# Function to scale the data to 0-255 range
def scale_data(data):
    min_value, max_value = np.percentile(data, [2, 98])
    scaled_data = (data - min_value) / (max_value - min_value) * 255
    scaled_data = np.clip(scaled_data, 0, 255)
    return scaled_data.astype(np.uint8)

# Function to convert data to decibel units and stretch within fixed thresholds
def convert_to_db_and_stretch(data, lower_threshold, upper_threshold):
    data_db = 10 * np.log10(data)
    data_db = (data_db - lower_threshold) / (upper_threshold - lower_threshold) * 255
    data_db = np.clip(data_db, 0, 255)
    return data_db.astype(np.uint8)

# Function to generate the mask library based on dB image and threshold values
def generate_mask_lib(db_image):
    # Define the threshold values for different ice types in dB
    threshold_ice_free = [2, 7]  # Ice Free threshold
    threshold_ice_covered = [-2, 3.5]  # Ice Covered threshold
    threshold_multiyearice = [-np.inf, -10]  # Multi-Year Ice threshold
    threshold_firstyearice = [-10, -14]  # First-Year Ice threshold
    threshold_icetype4 = [-14, -17]  # Ice Type 4 threshold
    threshold_newice = [-17, -22]  # New Ice threshold

    # Generate the mask library based on the dB image and threshold values
    mask_ice_free = (db_image >= threshold_ice_free[0]) & (db_image <= threshold_ice_free[1])
    mask_ice_covered = (db_image >= threshold_ice_covered[0]) & (db_image <= threshold_ice_covered[1])
    mask_multiyearice = (db_image >= threshold_multiyearice[0]) & (db_image <= threshold_multiyearice[1])
    mask_firstyearice = (db_image >= threshold_firstyearice[0]) & (db_image <= threshold_firstyearice[1])
    mask_icetype4 = (db_image >= threshold_icetype4[0]) & (db_image <= threshold_icetype4[1])
    mask_newice = (db_image >= threshold_newice[0]) & (db_image <= threshold_newice[1])

    mask_lib = {
        'Ice Free': mask_ice_free.astype(int),
        'Ice Covered': mask_ice_covered.astype(int),
        'Multi-Year Ice': mask_multiyearice.astype(int),
        'First-Year Ice': mask_firstyearice.astype(int),
        'Ice Type 4': mask_icetype4.astype(int),
        'New Ice': mask_newice.astype(int)
    }

    return mask_lib

# Function to resize the HV image to match the dimensions of the HH image
def resize_hv_data(hv_data, hv_transform, hh_transform, hh_shape):
    hv_data_resized = np.empty(hh_shape, dtype=hv_data.dtype)
    warp.reproject(
        source=hv_data,
        destination=hv_data_resized,
        src_transform=hv_transform,
        src_crs=hv_ds.crs,
        dst_transform=hh_transform,
        dst_crs=hh_ds.crs,
        resampling=Resampling.nearest
    )
    return hv_data_resized

# List the HH and HV file paths in the respective folders
hh_folder = 'd:/ISRO/Proj/HH_tiles'
hv_folder = 'd:/ISRO/Proj/HV_tiles'
output_folder = 'd:/ISRO/Proj/working/input'
os.makedirs(output_folder, exist_ok=True)

hh_files = os.listdir(hh_folder)
hv_files = os.listdir(hv_folder)

# Iterate over the files and process each pair of HH and HV images
for hh_file, hv_file in zip(hh_files, hv_files):
    hh_tif_file = os.path.join(hh_folder, hh_file)
    hv_tif_file = os.path.join(hv_folder, hv_file)

    with rasterio.open(hh_tif_file) as hh_ds:
        hh_data = hh_ds.read(1)

    with rasterio.open(hv_tif_file) as hv_ds:
        hv_data = hv_ds.read(1)
        hv_transform = hv_ds.transform

        # Resize the HV image to match the dimensions of the HH image
        hh_transform = hh_ds.transform
        hv_data_resized = resize_hv_data(hv_data, hv_transform, hh_transform, hh_data.shape)

    # Set zero and negative values in hv_data_resized to a small positive value to avoid warnings
    hv_data_resized[hv_data_resized <= 0] = 1e-9

    # Calculate the HH / HV ratio and HH - HV difference
    hh_hv_ratio = hh_data / hv_data_resized
    hh_hv_diff = hh_data - hv_data_resized

    # Convert the data to dB units using the 'convert_to_db_and_stretch' function
    hh_hv_ratio_db = convert_to_db_and_stretch(hh_hv_ratio, 2, 7)
    hh_hv_diff_db = convert_to_db_and_stretch(hh_hv_diff, -2, 3.5)

    # Create gray-scale composite by combining the processed HH-HV ratio and HH-HV difference
    gray_composite = 0.5 * hh_hv_ratio_db + 0.5 * hh_hv_diff_db

    # Save the gray-scale composite as a TIFF file
    output_tif_file = os.path.join('d:/ISRO/Proj/working/input', hh_file.replace('.tif', '_gray_composite.tif'))

    with rasterio.open(output_tif_file, 'w', driver='GTiff', width=hh_ds.width, height=hh_ds.height, count=1, dtype=gray_composite.dtype) as dst:
        dst.write(gray_composite, 1)

    print(f"Gray-scale composite image saved as {output_tif_file}")

/tmp/ipykernel_32/2798684523.py:17: RuntimeWarning: invalid value encountered in log10
  data_db = 10 * np.log10(data)
/opt/conda/lib/python3.10/site-packages/rasterio/__init__.py:314: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(


Gray-scale composite image saved as /kaggle/working/input/2354586151_SIGMANAUGHT_L2A_ORBIT-4399-LEVEL-STD-MODE-MRSHH_gray_composite.tif


/tmp/ipykernel_32/2798684523.py:17: RuntimeWarning: invalid value encountered in log10
  data_db = 10 * np.log10(data)
/opt/conda/lib/python3.10/site-packages/rasterio/__init__.py:314: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(


Gray-scale composite image saved as /kaggle/working/input/2354586141_SIGMANAUGHT_L2A_ORBIT-4399-LEVEL-STD-MODE-MRSHH_gray_composite.tif


/tmp/ipykernel_32/2798684523.py:17: RuntimeWarning: invalid value encountered in log10
  data_db = 10 * np.log10(data)
/opt/conda/lib/python3.10/site-packages/rasterio/__init__.py:314: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(


...

Gray-scale composite image saved as /kaggle/working/input/2354586181_SIGMANAUGHT_L2A_ORBIT-4273-LEVEL-STD-MODE-MRSHH_gray_composite.tif


/tmp/ipykernel_32/2798684523.py:17: RuntimeWarning: invalid value encountered in log10
  data_db = 10 * np.log10(data)
/opt/conda/lib/python3.10/site-packages/rasterio/__init__.py:314: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(


Gray-scale composite image saved as /kaggle/working/input/235458671_SIGMANAUGHT_L2A_ORBIT-5699-LEVEL-STD-MODE-MRS-POL-HH_gray_composite.tif


/tmp/ipykernel_32/2798684523.py:17: RuntimeWarning: invalid value encountered in log10
  data_db = 10 * np.log10(data)
/opt/conda/lib/python3.10/site-packages/rasterio/__init__.py:314: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(


Gray-scale composite image saved as /kaggle/working/input/235458611_SIGMANAUGHT_L2A_ORBIT-6228-LEVEL-STD-MODE-MRS-POL-HH_gray_composite.tif


In [ ]:
import os
import matplotlib.pyplot as plt
import tifffile as tiff

# Set the folder path containing the images
folder_path = "d:/ISRO/Proj/HH_tiles"

# Get a list of files in the folder
file_list = os.listdir(folder_path)

# Create a subplot grid to display the images
num_images = len(file_list)
num_rows = int(np.ceil(np.sqrt(num_images)))
num_cols = int(np.ceil(num_images / num_rows))

fig, axes = plt.subplots(num_rows, num_cols, figsize=(10, 10))

# Iterate over each file in the folder
for i, file_name in enumerate(file_list):
    # Check if the file is an image (TIFF format)
    if file_name.lower().endswith('.tif') or file_name.lower().endswith('.tif'):
        # Get the directory path of the image file
        file_path = os.path.join(folder_path, file_name)
        
        # Load and display the image using tifffile
        image = tiff.imread(file_path)
        
        # Determine the subplot indices
        row_idx = i // num_cols
        col_idx = i % num_cols
        
        # Display the image in the corresponding subplot
        axes[row_idx, col_idx].imshow(image, cmap='gray')
        axes[row_idx, col_idx].axis('off')

# Adjust the spacing between subplots
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
import os
import matplotlib.pyplot as plt
import tifffile as tiff

# Set the folder path containing the images
folder_path = "d:/ISRO/Proj/HV_tiles"

# Get a list of files in the folder
file_list = os.listdir(folder_path)

# Create a subplot grid to display the images
num_images = len(file_list)
num_rows = int(np.ceil(np.sqrt(num_images)))
num_cols = int(np.ceil(num_images / num_rows))

fig, axes = plt.subplots(num_rows, num_cols, figsize=(10, 10))

# Iterate over each file in the folder
for i, file_name in enumerate(file_list):
    # Check if the file is an image (TIFF format)
    if file_name.lower().endswith('.tif') or file_name.lower().endswith('.tif'):
        # Get the directory path of the image file
        file_path = os.path.join(folder_path, file_name)
        
        # Load and display the image using tifffile
        image = tiff.imread(file_path)
        
        # Determine the subplot indices
        row_idx = i // num_cols
        col_idx = i % num_cols
        
        # Display the image in the corresponding subplot
        axes[row_idx, col_idx].imshow(image, cmap='gray')
        axes[row_idx, col_idx].axis('off')

# Adjust the spacing between subplots
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
import os
import matplotlib.pyplot as plt
import tifffile as tiff

# Set the folder path containing the images
folder_path = "d:/ISRO/Proj/working/input"

# Get a list of files in the folder
file_list = os.listdir(folder_path)

# Create a subplot grid to display the images
num_images = len(file_list)
num_rows = int(np.ceil(np.sqrt(num_images)))
num_cols = int(np.ceil(num_images / num_rows))

fig, axes = plt.subplots(num_rows, num_cols, figsize=(10, 10))

# Iterate over each file in the folder
for i, file_name in enumerate(file_list):
    # Check if the file is an image (TIFF format)
    if file_name.lower().endswith('.tif') or file_name.lower().endswith('.tif'):
        # Get the directory path of the image file
        file_path = os.path.join(folder_path, file_name)
        
        # Load and display the image using tifffile
        image = tiff.imread(file_path)
        
        # Determine the subplot indices
        row_idx = i // num_cols
        col_idx = i % num_cols
        
        # Display the image in the corresponding subplot
        axes[row_idx, col_idx].imshow(image, cmap='gray')
        axes[row_idx, col_idx].axis('off')

# Adjust the spacing between subplots
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
import numpy as np
import cv2
import os

# Function to perform Min-Max normalization
def normalize_backscattering(data):
    min_value = np.min(data)
    max_value = np.max(data)
    normalized_data = 255 * (data - min_value) / (max_value - min_value)
    normalized_data = normalized_data.astype(np.uint8)
    return normalized_data

# Folder path containing the TIFF files
folder_path = 'd:/ISRO/Proj/working/input'  # Replace this with the actual path to your folder

# Output folder for the normalized TIFF files
norm_folder = 'd:/ISRO/Proj/working/norm_data'  # Replace this with the desired output path

# Check if the folder exists
if not os.path.exists(folder_path):
    print(f"Folder not found: {folder_path}")
else:
    # Create the output folder if it doesn't exist
    if not os.path.exists(norm_folder):
        os.makedirs(norm_folder)

    # Loop through each TIFF file in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.tiff') or filename.endswith('.tif'):
            # Read the TIFF file
            file_path = os.path.join(folder_path, filename)
            backscattering_data = cv2.imread(file_path, cv2.IMREAD_UNCHANGED)

            # Perform Min-Max normalization
            normalized_data = normalize_backscattering(backscattering_data)

            # Print the minimum and maximum values in the normalized backscattering data
            min_value = np.min(normalized_data)
            max_value = np.max(normalized_data)
            print(f"File: {filename}")
            print("Minimum value:", min_value)
            print("Maximum value:", max_value)
            print("\n")

            # Save the normalized data as a new TIFF file
            new_filename = "normalized_" + filename
            new_file_path = os.path.join(norm_folder, new_filename)
            cv2.imwrite(new_file_path, normalized_data)

print("Normalization and saving complete!")


In [ ]:
#This code is for composite image dataset
import numpy as np
from skimage.transform import resize
import os
import tifffile

def convert_to_db(image):
    # Replace 0 values with a small non-zero value (e.g., 1e-9)
    image[image == 0] = 1e-9

    # Clip the image array to avoid very large values before dB conversion
    min_clip_value = 1e-9
    max_clip_value = 1e9
    image = np.clip(image, min_clip_value, max_clip_value)

    with np.errstate(divide='ignore', invalid='ignore'):
        db_image = 10 * np.log10(image)
    db_image[np.isinf(db_image)] = np.nan
    return db_image


def generate_mask_lib(db_image):
    # Define the threshold values for different ice types in dB
    threshold_ice_free = [-90, -89.5]  # Ice Free threshold
    threshold_ice_bergs = [-89, -85]  # Ice Covered threshold
    threshold_multiyearice = [-85,-75]  # Multi-Year Ice threshold
    threshold_firstyearice = [-89.5, -89]  # First-Year Ice threshold

    # Generate the mask library based on the dB image and threshold values
    mask_ice_free = (db_image >= threshold_ice_free[0]) & (db_image <= threshold_ice_free[1])
    mask_ice_bergs = (db_image >= threshold_ice_bergs[0]) & (db_image <= threshold_ice_bergs[1])
    mask_multiyearice = (db_image >= threshold_multiyearice[0]) & (db_image <= threshold_multiyearice[1])
    mask_firstyearice = (db_image >= threshold_firstyearice[0]) & (db_image <= threshold_firstyearice[1])

    mask_lib = {
        'Ice Free': mask_ice_free.astype(int),
        'Ice bergs': mask_ice_bergs.astype(int),
        'Multi-Year Ice': mask_multiyearice.astype(int),
        'First-Year Ice': mask_firstyearice.astype(int),
    }

    return mask_lib

# Function to map backscattering values to label
def map_backscatter_to_label(backscatter):
    if -90 <= backscatter <= -89.5:
        return 0  # Ice Free
    elif -89 <= backscatter <= -85:
        return 1  # Ice bergs
    elif -85 <= backscatter <= -76:
        return 2  # Multi-Year Ice
    elif -89.5 <= backscatter <= -89:
        return 3  # First-Year Ice
    else:
        return -1  # Invalid label

# Folder path containing the SAR images (TIFF files)
folder_path = "d:/ISRO/Proj/working/norm_data"

# Initialize the lists to store the backscattering values and labels
backscatter_values = []
y = []
X = []

# Loop through the images in the folder
for filename in os.listdir(folder_path):
    # Load the SAR image (TIFF)
    image_path = os.path.join(folder_path, filename)
    image_array = tifffile.imread(image_path)

    # Check for invalid values in the image_array
    if np.any(np.isnan(image_array)) or np.any(np.isinf(image_array)):
        print(f"Invalid values found in {filename}. Skipping...")
        continue

    # Convert the SAR image to dB units
    db_image = convert_to_db(image_array)
    #print(f"Filename: {filename}, Decibel value: {db_image}")
    
    # Get the backscattering value (mean or other representative metric if needed)
    backscatter_value = np.nanmean(db_image)

    print(f"Filename: {filename}, Backscatter Value: {backscatter_value}")

    # Map the backscattering value to the label
    label_encoded = map_backscatter_to_label(backscatter_value)

    print(f"Encoded Label: {label_encoded}")

    # Append the encoded label to the list and continue if label is invalid (-1)
    if label_encoded != -1:
        y.append(label_encoded)

        # Resize the image to a consistent size
        resized_image = resize(image_array, (150, 150))

        X.append(resized_image)

# Convert the lists to arrays
X = np.array(X)
y = np.array(y)

# Print the final encoded labels
print("Encoded Labels:")
print(y)


In [ ]:
# #this code is for individual HH and HV and not for coomposite image dataset
# import numpy as np
# from skimage.transform import resize
# import os
# import tifffile

# def convert_to_db(image):
#     with np.errstate(divide='ignore', invalid='ignore'):
#         db_image = 10 * np.log10(image)
#     db_image[np.isinf(db_image)] = np.nan
#     return db_image

# def generate_mask_lib(db_image):
#     # Define the threshold values for different ice types in dB
#     threshold_ice_free = [-4.5, -3.5]  # Ice Free threshold
#     threshold_ice_bergs = [-3, -2.6]  # Ice Covered threshold
#     threshold_multiyearice = [-2.6,-2.3]  # Multi-Year Ice threshold
#     threshold_firstyearice = [-3.5, -3]  # First-Year Ice threshold

#     # Generate the mask library based on the dB image and threshold values
#     mask_ice_free = (db_image >= threshold_ice_free[0]) & (db_image <= threshold_ice_free[1])
#     mask_ice_bergs = (db_image >= threshold_ice_bergs[0]) & (db_image <= threshold_ice_bergs[1])
#     mask_multiyearice = (db_image >= threshold_multiyearice[0]) & (db_image <= threshold_multiyearice[1])
#     mask_firstyearice = (db_image >= threshold_firstyearice[0]) & (db_image <= threshold_firstyearice[1])

#     mask_lib = {
#         'Ice Free': mask_ice_free.astype(int),
#         'Ice bergs': mask_ice_bergs.astype(int),
#         'Multi-Year Ice': mask_multiyearice.astype(int),
#         'First-Year Ice': mask_firstyearice.astype(int),
#     }

#     return mask_lib

# # Function to map backscattering values to label
# def map_backscatter_to_label(backscatter):
#     if -4.5 <= backscatter <= -3.5:
#         return 0  # Ice Free
#     elif -3 <= backscatter <= -2.6:
#         return 1  # Ice bergs
#     elif -2.6 <= backscatter <= -2.3:
#         return 2  # Multi-Year Ice
#     elif -3.5 <= backscatter <= -3:
#         return 3  # First-Year Ice
#     else:
#         return -1  # Invalid label

# # Folder path containing the SAR images (TIFF files)
# folder_path = "d:/ISRO/Proj/working/norm_data"

# # Initialize the lists to store the backscattering values and labels
# backscatter_values = []
# y = []
# X = []

# # Loop through the images in the folder
# for filename in os.listdir(folder_path):
#     # Load the SAR image (TIFF)
#     image_path = os.path.join(folder_path, filename)
#     image_array = tifffile.imread(image_path)

#     # Check for invalid values in the image_array
#     if np.any(np.isnan(image_array)) or np.any(np.isinf(image_array)):
#         print(f"Invalid values found in {filename}. Skipping...")
#         continue

#     # Convert the SAR image to dB units
#     db_image = convert_to_db(image_array)
#     #print(f"Filename: {filename}, Decibel value: {db_image}")
    
#     # Get the backscattering value (mean or other representative metric if needed)
#     backscatter_value = np.nanmean(db_image)

#     print(f"Filename: {filename}, Backscatter Value: {backscatter_value}")

#     # Map the backscattering value to the label
#     label_encoded = map_backscatter_to_label(backscatter_value)

#     print(f"Encoded Label: {label_encoded}")

#     # Append the encoded label to the list and continue if label is invalid (-1)
#     if label_encoded != -1:
#         y.append(label_encoded)

#         # Resize the image to a consistent size
#         resized_image = resize(image_array, (150, 150))

#         X.append(resized_image)

# # Convert the lists to arrays
# X = np.array(X)
# y = np.array(y)

# # Perform any further processing or modeling with X and y as needed

# # Print the final encoded labels
# print("Encoded Labels:")
# print(y)


In [ ]:
print(y.shape)

In [ ]:
import numpy as np

# Assuming you have already loaded and processed the training data (x_train and z_train)

# Step 1: Print the unique values in z_train
unique_labels = np.unique(y)
print("Unique labels in z:", unique_labels)

# Step 2: Check if the labels are integers and within the correct range
num_classes = 4  # Assuming you have 4 classes (including the background class)
if np.issubdtype(y.dtype, np.integer):
    min_label = np.min(y)
    max_label = np.max(y)
    if min_label >= 0 and max_label <= (num_classes - 1):
        print("Labels are integers and within the correct range (0 to", num_classes - 1, ")")
    else:
        print("Labels are integers, but they are not within the correct range.")
        print("Minimum label:", min_label, ", Maximum label:", max_label)
else:
    print("Labels are not integers.")

In [ ]:
import numpy as np

# Assuming you have already loaded and processed the training data (x_train and z_train)

# Step 1: Print the unique values in z_train
unique_labels = np.unique(y)
print("Unique labels in z_train:", unique_labels)

# Step 2: Check if the labels are integers and within the correct range
num_classes = 4  # Assuming you have 4 classes (including the background class)
if np.issubdtype(y.dtype, np.integer):
    min_label = np.min(y)
    max_label = np.max(y)
    if min_label >= 0 and max_label <= (num_classes - 1):
        print("Labels are integers and within the correct range (0 to", num_classes - 1, ")")
    else:
        print("Labels are integers, but they are not within the correct range.")
        print("Minimum label:", min_label, ", Maximum label:", max_label)
else:
    print("Labels are not integers.")


In [ ]:
# Assuming num_classes = 4 (including the background class)
num_classes = 4

# Rescale the integer labels to be within the correct range (0 to num_classes - 1)
z_train_int_rescaled = (y / 255) * (num_classes - 1)

# Convert the rescaled labels back to integers
z_train_int_rescaled = np.round(z_train_int_rescaled).astype(int)

# Verify the unique labels and check if they are integers and within the correct range
unique_labels_int_rescaled = np.unique(z_train_int_rescaled)

min_label_rescaled = np.min(z_train_int_rescaled)
max_label_rescaled = np.max(z_train_int_rescaled)

if np.issubdtype(z_train_int_rescaled.dtype, np.integer):
    if min_label_rescaled >= 0 and max_label_rescaled <= (num_classes - 1):
        print("Labels are integers and within the correct range (0 to", num_classes - 1, ")")
    else:
        print("Labels are integers, but they are not within the correct range.")
        print("Minimum label:", min_label_rescaled, ", Maximum label:", max_label_rescaled)
else:
    print("Labels are not integers.")

In [ ]:
#Augmentation
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from skimage.transform import resize

# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=5,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

import os

# Define the directory to save the augmented images
save_dir = 'd:/ISRO/Proj/working/augmented_img'

# Create the directory if it doesn't exist
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# Create augmented images and labels
print('Performing data augmentation')
augmented_images = []
augmented_labels = []
num_augmented_samples = 3  # Number of augmented samples to generate per original image

for i in range(len(X)):
    img = X[i]
    label = y[i]

    # Reshape the image to include the channel dimension
    img = np.expand_dims(img, axis=-1)

    # Generate augmented samples
    img_augmented_gen = datagen.flow(np.expand_dims(img, axis=0), batch_size=1, shuffle=False)

    for j in range(num_augmented_samples):
        img_augmented = img_augmented_gen.next()[0]

        # Resize the augmented image to a consistent size
        img_augmented = resize(img_augmented, (150, 150))

        augmented_images.append(img_augmented)
        augmented_labels.append(label)

        # Save the augmented image
        img_save_path = os.path.join(save_dir, f'image_{i}_{j}.png')
        plt.imsave(img_save_path, img_augmented.squeeze(), cmap='gray')

# Convert the lists to arrays
augmented_images = np.array(augmented_images)
augmented_labels = np.array(augmented_labels)

augmented_labels = np.expand_dims(augmented_labels, axis=-1)


print('Augmented images shape:', augmented_images.shape)
print('Augmented labels shape:', augmented_labels.shape)
print('Augmented images saved in:', save_dir)

In [ ]:
# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(augmented_images, augmented_labels, test_size=0.2, random_state=42)

In [ ]:
unique_classes = np.unique(y_train)
print("Unique Classes:", unique_classes)

In [ ]:
# Print the shapes of X_train, X_test, y_train, y_test
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

In [ ]:
#build classification model

from sklearn.metrics import accuracy_score
from sklearn.metrics import matthews_corrcoef
from sklearn.metrics import f1_score

In [ ]:
unique_classes = np.unique(y_train)
print("Unique Classes:", unique_classes)

In [ ]:
unique_labels = np.unique(y_train)
print(unique_labels)

In [ ]:
import numpy as np
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from keras.utils import to_categorical
from scikeras.wrappers import KerasClassifier

# Reshape the training data
n_samples_train = X_train.shape[0]
n_channels_train = X_train.shape[3]
n_rows_train = X_train.shape[1]
n_cols_train = X_train.shape[2]
X_train_2d = X_train.reshape(n_samples_train, n_rows_train, n_cols_train, n_channels_train)

# Reshape the test data
n_samples_test = X_test.shape[0]
n_channels_test = X_test.shape[3]
n_rows_test = X_test.shape[1]
n_cols_test = X_test.shape[2]
X_test_2d = X_test.reshape(n_samples_test, n_rows_test, n_cols_test, n_channels_test)

# Calculate the number of unique classes in your dataset
n_classes = len(np.unique(y_train))

# Convert y_train to one-hot encoded format
y_train_categorical = to_categorical(y_train, num_classes=n_classes)

# Convert y_test to one-hot encoded format
y_test_categorical = to_categorical(y_test, num_classes=n_classes)

def cnn():
    # Define the model
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(n_rows_train, n_cols_train, n_channels_train)))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dense(n_classes, activation='softmax'))  # Update the number of neurons here

    # Compile the model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = KerasClassifier(cnn)

# Train the model
model.fit(X_train_2d, y_train_categorical, epochs=50, batch_size=32)

# Make predictions
y_train_pred = model.predict(X_train_2d)
y_test_pred = model.predict(X_test_2d)

# Evaluate the model on the training data
train_accuracy = model.score(X_train_2d, y_train_categorical)
print('Training Accuracy:', train_accuracy)

# Evaluate the model on the test data
test_accuracy = model.score(X_test_2d, y_test_categorical)
print('Test Accuracy:', test_accuracy)


In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import matthews_corrcoef, f1_score

# Reshape the training data
n_samples_train = X_train.shape[0]
n_features_train = np.prod(X_train.shape[1:])
X_train_2d = X_train.reshape(n_samples_train, n_features_train)

# Reshape the test data
n_samples_test = X_test.shape[0]
n_features_test = np.prod(X_test.shape[1:])
X_test_2d = X_test.reshape(n_samples_test, n_features_test)

# Ensure consistent number of samples between features and targets
y_train_flat = y_train[:n_samples_train].ravel()
y_test_flat = y_test[:n_samples_test].ravel()

# Define the SVM model
class SVMClassifier(SVC):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def fit(self, X, y):
        n_samples = X.shape[0]
        n_features = np.prod(X.shape[1:])
        X_2d = X.reshape(n_samples, n_features)
        y_flat = y.ravel()
        return super().fit(X_2d, y_flat)

    def predict(self, X):
        n_samples = X.shape[0]
        n_features = np.prod(X.shape[1:])
        X_2d = X.reshape(n_samples, n_features)
        return super().predict(X_2d)

# Define the SVM classifier
svm_rbf = SVMClassifier(kernel='rbf', gamma=2, C=1)

# Fit the SVM model
svm_rbf.fit(X_train_2d, y_train_flat)

# Define the estimators list
estimator_list = [
    ('svm', SVMClassifier(gamma='scale'))
]

# Instantiate the VotingClassifier with 'hard' voting
voting_classifier = VotingClassifier(estimators=estimator_list, voting='hard')

# Fit the VotingClassifier
voting_classifier.fit(X_train_2d, y_train_flat)

# Make predictions
y_train_pred = voting_classifier.predict(X_train_2d)
y_test_pred = voting_classifier.predict(X_test_2d)

# Calculate MCC and F1-score for training set
train_mcc = matthews_corrcoef(y_train_flat, y_train_pred)
train_f1_macro = f1_score(y_train_flat, y_train_pred, average='macro')
train_f1_micro = f1_score(y_train_flat, y_train_pred, average='micro')

# Calculate MCC and F1-score for test set
test_mcc = matthews_corrcoef(y_test_flat, y_test_pred)
test_f1_macro = f1_score(y_test_flat, y_test_pred, average='macro')
test_f1_micro = f1_score(y_test_flat, y_test_pred, average='micro')

# Print the MCC and F1-score
print('Training set:')
print('MCC:', train_mcc)
print('F1-score (macro):', train_f1_macro)
print('F1-score (micro):', train_f1_micro)
print('------------------------')
print('Test set:')
print('MCC:', test_mcc)
print('F1-score (macro):', test_f1_macro)
print('F1-score (micro):', test_f1_micro)


In [ ]:
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score

# Reshape the training data
n_samples_train = X_train.shape[0]
n_features_train = np.prod(X_train.shape[1:])
X_train_2d = X_train.reshape(n_samples_train, n_features_train)

# Reshape the test data
n_samples_test = X_test.shape[0]
n_features_test = np.prod(X_test.shape[1:])
X_test_2d = X_test.reshape(n_samples_test, n_features_test)

# Flatten the target variable
y_train_flat = y_train.ravel()
y_test_flat = y_test.ravel()

# Define the MLPClassifier model
mlp = MLPClassifier(alpha=1, max_iter=1000, random_state=42)  # Add random_state for reproducibility

# Fit the MLPClassifier model
mlp.fit(X_train_2d, y_train_flat)

# Make predictions
y_train_pred = mlp.predict(X_train_2d)
y_test_pred = mlp.predict(X_test_2d)

# Training set performance
mlp_train_accuracy = accuracy_score(y_train_flat, y_train_pred)  # Calculate Accuracy
mlp_train_mcc = matthews_corrcoef(y_train_flat, y_train_pred)  # Calculate MCC
mlp_train_f1 = f1_score(y_train_flat, y_train_pred, average='weighted')  # Calculate F1-score

# Test set performance
mlp_test_accuracy = accuracy_score(y_test_flat, y_test_pred)  # Calculate Accuracy
mlp_test_mcc = matthews_corrcoef(y_test_flat, y_test_pred)  # Calculate MCC
mlp_test_f1 = f1_score(y_test_flat, y_test_pred, average='weighted')  # Calculate F1-score

print('Model performance for Training set')
print('- Accuracy:', mlp_train_accuracy)
print('- MCC:', mlp_train_mcc)
print('- F1 score:', mlp_train_f1)
print('----------------------------------')
print('Model performance for Test set')
print('- Accuracy:', mlp_test_accuracy)
print('- MCC:', mlp_test_mcc)
print('- F1 score:', mlp_test_f1)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import itertools
from sklearn.ensemble import StackingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score, confusion_matrix
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Generate some random data for demonstration
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define base classifiers
svm = SVC()
mlp = MLPClassifier()

# Define estimators
estimator_list = [('svm_rbf', svm), ('mlp', mlp)]

# Build stack model
stack_model = StackingClassifier(estimators=estimator_list, final_estimator=LogisticRegression())

# Train stacked model
stack_model.fit(X_train, y_train)

# Make predictions
y_train_pred = stack_model.predict(X_train)
y_test_pred = stack_model.predict(X_test)

# Training set model performance
stack_model_train_accuracy = accuracy_score(y_train, y_train_pred)
stack_model_train_mcc = matthews_corrcoef(y_train, y_train_pred)
stack_model_train_f1 = f1_score(y_train, y_train_pred, average='weighted')

# Test set model performance
stack_model_test_accuracy = accuracy_score(y_test, y_test_pred)
stack_model_test_mcc = matthews_corrcoef(y_test, y_test_pred)
stack_model_test_f1 = f1_score(y_test, y_test_pred, average='weighted')

print('Model performance for Training set')
print('- Accuracy:', stack_model_train_accuracy)
print('- MCC:', stack_model_train_mcc)
print('- F1 score:', stack_model_train_f1)
print('----------------------------------')
print('Model performance for Test set')
print('- Accuracy:', stack_model_test_accuracy)
print('- MCC:', stack_model_test_mcc)
print('- F1 score:', stack_model_test_f1)


# Function to plot confusion matrix
def plot_confusion_matrix(cm, classes, normalize=False, title='Confusion Matrix', cmap=plt.cm.Blues):
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized Confusion Matrix")
    else:
        print('Confusion Matrix, without Normalization')

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt), horizontalalignment="center", color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')


# Calculate confusion matrix for training set
train_confusion_matrix = confusion_matrix(y_train, y_train_pred)
plt.figure(figsize=(10, 8))
plot_confusion_matrix(train_confusion_matrix, classes=np.unique(y_train), normalize=False, title="Confusion Matrix (Training Set)")
plt.show()

# Calculate confusion matrix for test set
test_confusion_matrix = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(10, 8))
plot_confusion_matrix(test_confusion_matrix, classes=np.unique(y_test), normalize=False, title="Confusion Matrix (Test Set)")
plt.show()


In [ ]:
#Results
acc_train_list = {
#'svm_rbf': svm_rbf_train_accuracy,
'cnn': mlp_train_accuracy,
'stack': stack_model_train_accuracy}

mcc_train_list = {
'svm_rbf': train_mcc,
'mlp': mlp_train_mcc,
'stack': stack_model_train_mcc}

f1_train_list = {
'svm_rbf': train_f1_micro,
'mlp': mlp_train_f1,
'stack': stack_model_train_f1}

In [ ]:
acc_train_list

In [ ]:
mcc_train_list

In [ ]:
f1_train_list

In [ ]:
import pandas as pd

acc_df = pd.DataFrame.from_dict(acc_train_list, orient='index', columns=['Accuracy'])
mcc_df = pd.DataFrame.from_dict(mcc_train_list, orient='index', columns=['MCC'])
f1_df = pd.DataFrame.from_dict(f1_train_list, orient='index', columns=['F1'])
df = pd.concat([acc_df, mcc_df, f1_df], axis=1)

In [ ]:
df.to_csv('results.csv')